# kube-apiserver Attack Reproduction

## 1. Mechanism overview

This section details the mechanism described in the paper and the figures it reports, so they can be directly compared to our own results in section 3.

### 1.1 The default service account token as an attack surface

The paper describes a set of kube-apiserver attack strategies exploiting the default service account token, automatically mounted inside every pod at `/var/run/secrets/kubernetes.io/serviceaccount/token` (§3.5). RBAC restricts which interfaces this token can reach, but not how often it can be used to reach them. Table 4 lists a fixed set of 14 interfaces reachable with the default token regardless of the pod's own RBAC configuration, including `/healthz`, `/livez`, `/readyz`, `/version`, `/api`, `/apis`, `/openapi`, and the selfsubjectreview/selfsubjectaccessreview/selfsubjectrulesreview endpoints. Because none of these interfaces are rate limited, any pod in the cluster, however unprivileged, can send an unbounded volume of authenticated requests against them.

The strategy reproduced here corresponds to the paper's Case 2 (§4.3), where a small number of malicious containers hold open many connections against one of these allowed interfaces at once.

### 1.2 Two consequences of the same connection flood

The paper describes the attack as producing two effects at once:

1. System-level saturation via `nf_conntrack`. Sending a large number of TCP connections to any allowed interface fills the kernel's connection tracking table quickly. This effect depends only on the volume and persistence of connections reaching the apiserver, not on which interface or what content is requested: a conntrack entry is created as soon as the TCP handshake completes at the kernel level, independently of whether the TLS handshake or the HTTP request above it ever finishes.

2. CPU amplification via the specific interface queried. The interface chosen still matters for a second effect: `/healthz`, for instance, makes the kube-apiserver query other control-plane components internally as part of answering the health check (for instance etcd's own health key), so a flood of `/healthz` requests does not only stress the apiserver itself but propagates CPU load onto the components it queries in turn.

### 1.3 What the paper reports as impact

For the kube-apiserver attack (Case 2) in the local environment, the paper reports that 5 malicious containers fill the nf_conntrack table within 5 minutes, that the victim container suffers 30% packet loss when connecting to the control plane, and that the kube-apiserver together with the other control-plane components it queries consume 90% additional CPU (§4.3). These are the reference figures our own measurements are compared against in section 3, taken from the paper's local environment to match our own setup.

## 2. Reproduction: manifests and methodology

### 2.1 Attack topology

### 2.2 Final manifest

The attack deployment (`apiserver-flood.yaml`, `replicas: 5`, matching the paper's Case 2 setup) runs a Python asyncio program in each attacker pod. The full manifest is in the `manifests/` folder; the key parts are discussed below.

**Authentication with the default service account token.** Each connection authenticates with the token automatically mounted in every pod (section 1.1), read from disk and sent as a bearer token:

```python
TOKEN_PATH = "/var/run/secrets/kubernetes.io/serviceaccount/token"
with open(TOKEN_PATH) as f:
    TOKEN = f.read().strip()
req = (
    "GET /healthz HTTP/1.1\r\n"
    "Host: kubernetes.default.svc\r\n"
    f"Authorization: Bearer {TOKEN}\r\n"
    "Connection: keep-alive\r\n\r\n"
).encode()
```

`/healthz` is the queried interface because it is reachable with the default token regardless of the pod's RBAC (Table 4) and triggers the CPU amplification of section 1.2. `Connection: keep-alive` is what makes each connection reusable rather than one-shot.

**Connections are held open, not cycled.** Each of the `CONNS` connections is opened once and kept alive with a periodic re-ping, rather than opened and closed in a loop, so it holds exactly one conntrack entry for as long as it survives (the reason is detailed in section 2.3):

```python
async def hold():
    while True:
        try:
            reader, writer = await asyncio.open_connection(
                TARGET_IP, 443, ssl=ctx, server_hostname="kubernetes.default.svc")
            while True:
                writer.write(req); await writer.drain()
                await reader.read(4096)
                await asyncio.sleep(random.uniform(20, 60))   # re-ping interval
        except Exception:
            await asyncio.sleep(random.uniform(1, 6))          # retry backoff
```

Both delays are randomized rather than fixed, so connections established in the same burst do not re-ping or retry in a synchronized wave against the apiserver (section 2.3).

**Targeting the ClusterIP, not the hostname.** The apiserver is reached by its ClusterIP (`10.96.0.1`) directly, set through an environment variable:

```yaml
env:
- name: TARGET_IP
  value: "10.96.0.1"   # kubernetes service ClusterIP, avoids a DNS lookup per connection
- name: CONNS
  value: "28000"       # connections held open per pod
```

Connecting by the hostname `kubernetes.default.svc` instead would trigger a fresh DNS lookup on every new connection attempt, which under load generated far more conntrack entries (UDP, port 53, toward CoreDNS) than the intended TCP flood itself, confounding the measurement of the apiserver's own load. The hostname is still sent in the TLS SNI and HTTP `Host` header (required for the request to be served), but the TCP connection itself goes straight to the IP.

**Scale and file-descriptor ceiling.** The deployment runs `replicas: 5` pods, each holding up to `CONNS` connections, so the file-descriptor limit is raised before launching Python:

```sh
ulimit -n 32000
```

Without this, each pod would hit the default open-file limit long before reaching its share of the 131,072 target (section 2.3).

**The attack runs under the `restricted` Pod Security Standard.** The pods carry the hardened securityContext that `safe-ns` and `attacker-ns` enforce (section 2.1), with no privilege, no root, and all capabilities dropped:

```yaml
securityContext:
  allowPrivilegeEscalation: false
  privileged: false
  runAsNonRoot: true
  runAsUser: 1000
  seccompProfile:
    type: RuntimeDefault
  capabilities:
    drop: ["ALL"]
```

None of these restrictions prevent the attack: it needs only an outbound TLS connection and the service account token mounted by default, neither of which requires any elevated privilege. This is the point of the paper's threat model, that an entirely unprivileged pod is sufficient.

### 2.3 Why this design, and not an earlier one

**An initial single-source-IP hypothesis, confirmed.** Because all 5 attacker pods run on the same node, Calico's default `natOutgoing` setting masquerades their outbound traffic to the apiserver behind that node's single IP, which would cap the total number of concurrent connections at what a single IP's ephemeral port range allows (roughly 28,000 ports). This matched what was observed: with `natOutgoing` left enabled, the attack could not reach its intended saturation level. `natOutgoing` was disabled (`false`) so each attacker pod keeps its own distinct source IP, and it has been kept disabled since.

**Open/close connection churn versus held-open connections.** The very first version of the attack reused the thread-based pattern already working for the CoreDNS attack: several threads, each opening a fresh connection, sending a request, closing it, and looping, just adapted from a raw UDP socket to a TCP `HTTPSConnection` for the apiserver. This pattern does not transfer from UDP to TCP. A UDP send has no real connection to close, so a closed UDP flow's conntrack entry expires quickly (on the order of tens of seconds), and a fast loop of many threads keeps recreating entries about as fast as they expire, which is enough to sustain a high count. A closed TCP connection, on the other hand, does not free its conntrack entry right away: it sits in the kernel's TIME_WAIT state for a fixed duration of around 120 seconds first. This turns the table's size into a balance between how fast new connections are created and this fixed 120-second expiry delay, rather than something the loop's speed alone can push arbitrarily high: past the first 120 seconds, connections start expiring at roughly the same rate new ones are opened, so the count settles into a plateau instead of continuing to climb. This is what motivated the switch to connections that are established once and then kept alive: a held-open connection never enters TIME_WAIT while it survives, so it occupies its conntrack entry directly for as long as the attack runs, without being capped by this expiry delay.

**Fixed intervals versus jittered intervals.** With a fixed re-ping interval and a fixed retry backoff, connections established in the same initial burst tended to re-ping, and later retry, at nearly the same moment, producing a synchronized wave of requests against an apiserver already under load. This caused a pronounced sawtooth pattern: the table would saturate, a shared stress event (CPU spike, packet drops once the table is full) would cause many connections to fail at once, the table would partially drain, and the cycle would repeat once the synchronized retries succeeded again. Randomizing both the re-ping interval and the retry backoff spreads this recovery over time instead of concentrating it, which markedly reduced, though did not entirely eliminate, this oscillation: the retry backoff window (`random.uniform(1, 6)` seconds) was tuned as an explicit middle ground between a tighter window (which still triggered visible collapses) and a wider one (which slowed the refill enough to shorten how long the attack spent near saturation).

**Tuning `CONNS` to reach the ceiling without breaking the measurement pipeline.** With `nf_conntrack_max` fixed at 131,072, the target of roughly 26,214 held connections per pod (131,072 divided across 5 pods) is what the deployment aims to sustain. A `CONNS` value that undershoots this target plateaus well below the ceiling; a value that overshoots it by too much saturates the apiserver so completely that even the measurement tooling itself (a `kubectl exec`-based TCP probe, or `kubectl exec` used for any other purpose) can no longer get a response, which defeats the point of measuring packet loss at all. In practice, the right value is not a single fixed number: it depends on the state of the machine running the attacker pods at a given moment, for instance whether a previous run has left residual load behind, so `CONNS` is adjusted slightly from one run to the next (28,000 per pod as a starting point, lowered when a prior run's effects have not fully settled) rather than pinned to one definitive value.

### 2.4 Measurement scripts

Four scripts drive the measurement, kept separate from the attack manifest itself. The measurement is deliberately split by node so that no probe depends on the component under attack.

**`launch-attack2.sh`** deploys the attack (`apiserver-flood.yaml`) with no metrics collection, and runs on k8s-cp. Unlike CoreDNS, the kube-apiserver attack has no Prometheus counter that needs resetting between runs (the apiserver's `/metrics` endpoint is not reachable with the default service account token, so it is not part of this attack's measurement), so no control-plane restart is required before launching.

**`capture-cpu-loop-apiserver.sh`** measures the CPU usage of the `kube-apiserver` and `etcd` processes specifically (not whole-node CPU), run on k8s-cp. It reads `/proc/<pid>/stat` at the start and end of each interval and computes the CPU-tick delta, summing user and system ticks across every matching process. etcd is measured alongside the apiserver because the attack's CPU amplification propagates onto it (section 1.2). This is the per-process reading compared against the paper's "90% additional CPU" figure.

**`capture-tcp-loss-loop-nsenter.sh`** measures packet loss from the victim pod, run on k8s-w2. Rather than a `kubectl exec`-based probe, it enters the victim pod's own network namespace with `nsenter` (preserving its real source IP) and issues the TCP probe to the control plane directly, without involving the apiserver at all. This matters for this specific attack: `kubectl exec` is itself an API call routed through the kube-apiserver, the very component under attack, so a `kubectl exec`-based probe would conflate two different things; whether the victim can reach the control plane over TCP, and whether the control plane can still service an exec request. Only `OK` samples are averaged, since a failed probe is a measurement worth keeping, not a zero-valued reading.

**`watch-apiserver-metrics.sh`** (k8s-cp) reads `nf_conntrack_count`/`nf_conntrack_max` and node CPU (`%us`/`%sy`) locally, with no dependency on the apiserver, printing one line per interval. Its output is captured to a CSV with `tee` during the run, and is the source of the nf_conntrack occupancy figures in section 3.

One helper script supports these without feeding the results: **`tcp-latency-test-nsenter.sh`** (k8s-w2) is a standalone one-shot version of the nsenter probe, used to spot-check connectivity at any point during an attack.

## 3. Results

In [4]:
from aggregate_averages_apiserver import (
    find_watch_file,
    find_cpu_loop_files,
    find_tcp_loss_files,
    load_watch_rows,
    load_cpu_loop_rows,
    load_tcp_ok_rows,
    COLS,
    CPU_LOOP_COLS,
    TCP_COL,
    LABELS,
)
import pandas as pd
from IPython.display import display, HTML

def load_combined(label):
    watch_files = find_watch_file(label, base_dir="../results")
    cpu_loop_files = find_cpu_loop_files(label, base_dir="../results")
    tcp_files = find_tcp_loss_files(label, base_dir="../results")

    watch = pd.concat([load_watch_rows(f) for f in watch_files], ignore_index=True) if watch_files else pd.DataFrame(columns=COLS)
    cpu_loop = pd.concat([load_cpu_loop_rows(f) for f in cpu_loop_files], ignore_index=True) if cpu_loop_files else pd.DataFrame(columns=CPU_LOOP_COLS)
    tcp = pd.concat([load_tcp_ok_rows(f) for f in tcp_files], ignore_index=True) if tcp_files else pd.DataFrame(columns=[TCP_COL])

    return watch, cpu_loop, tcp

before_watch, before_cpu_loop, before_tcp = load_combined("before")
after_watch, after_cpu_loop, after_tcp = load_combined("after")

rows = COLS + CPU_LOOP_COLS + [TCP_COL]
summary = pd.DataFrame({
    "Before": pd.concat([before_watch[COLS].mean(), before_cpu_loop[CPU_LOOP_COLS].mean(), before_tcp[[TCP_COL]].mean()]),
    "After": pd.concat([after_watch[COLS].mean(), after_cpu_loop[CPU_LOOP_COLS].mean(), after_tcp[[TCP_COL]].mean()]),
}).round(1).reindex(rows).rename(index=LABELS)

display(HTML("<h2>Metrics comparison before / after attack</h2>"))
display(summary.style.set_properties(**{"font-size": "16pt"}).set_table_styles(
    [{"selector": "th", "props": [("font-size", "16pt")]}]
))

,Before,After
nf_conntrack_count,328.500000,128534.700000
"CPU user, whole node (%)",3.700000,5.100000
"CPU system, whole node (%)",3.400000,6.100000
CPU kube-apiserver process (%),8.700000,33.400000
CPU etcd process (%),4.100000,2.300000
CPU kube-apiserver + etcd (%),12.800000,35.700000
TCP packet loss to control plane (%),0.000000,47.300000


## 3. Results

| Metric | Paper, local env. (§4.3) | Our reproduction |
|---|---|---|
| Time to fill nf_conntrack | ~5 minutes (5 containers) | under 2 minutes on average, faster than the paper |
| Packet loss on the victim container | 30% | mean 47.3% over 15 samples, but intermittent (0% to 100% range), not a steady rate |
| CPU consumed by kube-apiserver | 90% additional (jointly apiserver + components it queries) | 8.7% to 33.4%, a 24.7 point increase (apiserver + etcd only, narrower scope) |
| CPU consumed by etcd | included in the 90% figure above | 4.1% to 2.3%, a 1.8 point decrease |

Four things stand out from this comparison.

**nf_conntrack saturation is confirmed, and reached faster than in the paper's local reproduction.** The paper reports the table filling up within 5 minutes with 5 containers in its local environment. We timed this by starting each capture as the table began to fill: across our runs, full saturation was reached in under 2 minutes on average, faster than the paper's local figure. Once reached, saturation is sustained for nearly the whole attack: 98.1% mean occupancy (from `watch-apiserver-metrics.sh`, captured with `tee`), with most samples sitting at exactly 131,072/131,072 and occasional partial drains down to around 98,000 to 120,000.

**Packet loss on the victim container is higher than the paper's local figure, but highly intermittent.** The paper reports 30% packet loss in its local environment; our mean is 47.3%, but that mean hides the real behavior. Across our 15 attack-window samples (`tcp_probe_status == OK` only), loss is bimodal rather than centered: 6 samples show 0% loss (all 20 probes succeeding) while 3 show 100% (all failing), with the rest scattered between. The victim alternates between reaching the control plane normally and being entirely cut off, rather than suffering a steady partial loss. This tracks the conntrack behavior directly: when the table sits at its ceiling the probes fail, and during the partial drains seen in the watch capture (down to ~98,000 to 120,000) they succeed again, which is the same sawtooth described in section 2.3 read from the victim's side. The 47.3% mean is therefore an average over two opposite regimes, not a stable loss rate, and it rests on a small sample (15 points over roughly two minutes of probing), so it should be read as "frequently and severely degraded" rather than as a precise figure.

**kube-apiserver CPU consumption is measured against the process directly, not the whole node.** `cpu_kube_apiserver` and `cpu_etcd` (read from `/proc/<pid>/stat`, section 2.4) are compared against the paper's figure rather than the node-wide `top` reading. `kube-apiserver` rises from 8.7% to 33.4%, a 24.7 point increase. This sits well below the paper's 90%, but the two are not measured over the same scope: the paper's 90% is a composite attributed to "the kube-apiserver together with the other control-plane components it queries" (section 1.3), whereas our figure covers only the kube-apiserver and etcd processes. We measured those two because they are the ones section 1.2 identifies as directly on the `/healthz` path; the paper's figure additionally includes whatever other components its health checks reach, which we did not instrument. The gap is therefore primarily one of scope, not of attack intensity. A secondary, unconfirmed factor is scale: as with the CoreDNS attack, the paper does not state whether its CPU figures are normalized across cores (capped at 100%) or can exceed it like our per-process reading.

**etcd CPU consumption decreases during the attack, so the etcd component of the paper's CPU amplification is not reproduced here.** Section 1.2 describes the apiserver propagating load onto etcd when answering `/healthz` checks, and the paper folds etcd into its 90% figure. Our measurement shows the opposite: `etcd` CPU falls from 4.1% to 2.3% (a 1.8 point decrease), reproducibly across separate before/after runs. The absence of any etcd increase is consistent with what `/healthz` actually does: it triggers only a lightweight health read on etcd, not a write, so a flood of health checks adds little etcd load on its own. That accounts for etcd not rising; it does not by itself explain why etcd drops below its at-rest level. One possible reason, observed but not confirmed by this data, is that the apiserver's own saturation delays or drops the legitimate cluster traffic that normally reaches etcd (kubelet heartbeats, controller watches), pulling etcd's baseline load down. The decrease is reported as observed; the explanation for it is not established.